# Bundestag API – Exploration
Direkte HTTP-Calls gegen `https://search.dip.bundestag.de/api/v1`

In [37]:
import requests
import json
import pprint

BASE_URL = "https://search.dip.bundestag.de/api/v1"
API_KEY  = "R2BZaee.DjdCyihKZMf8AOjtScubP2EVydegzjmBIQ"  # öffentlicher Demo-Key

session = requests.Session()
session.headers.update({
    "Authorization": f"ApiKey {API_KEY}",
    "Accept": "application/json",
})

print("Session bereit.")

Session bereit.


## Plenarprotokolle abrufen

In [38]:
url = f"{BASE_URL}/plenarprotokoll"

params = {
    "wahlperiode": 20,
    "format": "json",
    "num": 5,
    "datum.start": "2024-01-01",
}

response = session.get(url, params=params, timeout=15)
print(f"Status : {response.status_code}")
print(f"URL    : {response.url}")

Status : 200
URL    : https://search.dip.bundestag.de/api/v1/plenarprotokoll?wahlperiode=20&format=json&num=5&datum.start=2024-01-01


In [39]:
data = response.json()

print(f"Anzahl Dokumente : {len(data.get('documents', []))}")
print(f"Cursor (next)    : {data.get('cursor', '–')}")
print()
print("Antwort-Keys:", list(data.keys()))

Anzahl Dokumente : 100
Cursor (next)    : AoJwuJCB3pQDNFBsZW5hcnByb3Rva29sbC01Njk3

Antwort-Keys: ['numFound', 'documents', 'cursor']


In [40]:
# Erstes Dokument im Detail
erstes = data["documents"][0]
pprint.pprint(erstes)

{'aktualisiert': '2026-05-19T12:13:55+02:00',
 'datum': '2026-05-22',
 'dokumentart': 'Plenarprotokoll',
 'dokumentnummer': '21/81',
 'fundstelle': {'datum': '2026-05-22',
                'dokumentart': 'Plenarprotokoll',
                'dokumentnummer': '21/81',
                'herausgeber': 'BT',
                'id': '5796',
                'pdf_url': 'https://dserver.bundestag.de/btp/21/21081.pdf',
                'urheber': []},
 'herausgeber': 'BT',
 'id': '5796',
 'titel': 'Protokoll der 81. Sitzung des 21. Deutschen Bundestages',
 'typ': 'Dokument',
 'vorgangsbezug_anzahl': 0,
 'wahlperiode': 21}


In [41]:
# Alle Dokumente tabellarisch
for dok in data["documents"]:
    print(f"{dok.get('datum', '?'):12}  id={dok.get('id', '?'):<8}  {dok.get('titel', '')[:80]}")

2026-05-22    id=5796      Protokoll der 81. Sitzung des 21. Deutschen Bundestages
2026-05-21    id=5795      Protokoll der 80. Sitzung des 21. Deutschen Bundestages
2026-05-20    id=5794      Protokoll der 79. Sitzung des 21. Deutschen Bundestages
2026-05-08    id=5793      Protokoll der 1065. Sitzung des Bundesrates
2026-05-08    id=5792      Protokoll der 78. Sitzung des 21. Deutschen Bundestages
2026-05-07    id=5791      Protokoll der 77. Sitzung des 21. Deutschen Bundestages
2026-05-06    id=5790      Protokoll der 76. Sitzung des 21. Deutschen Bundestages
2026-04-24    id=5789      Protokoll der 1064. Sitzung des Bundesrates
2026-04-24    id=5788      Protokoll der 75. Sitzung des 21. Deutschen Bundestages
2026-04-23    id=5787      Protokoll der 74. Sitzung des 21. Deutschen Bundestages
2026-04-22    id=5786      Protokoll der 73. Sitzung des 21. Deutschen Bundestages
2026-04-17    id=5785      Protokoll der 72. Sitzung des 21. Deutschen Bundestages
2026-04-16    id=5784      P

## Weitere Endpunkte

In [ ]:
def call_api(endpoint: str, **params) -> dict:
    """Hilfsfunktion: GET gegen einen beliebigen Endpunkt."""
    defaults = {"wahlperiode": 20, "format": "json", "num": 5}
    r = session.get(f"{BASE_URL}/{endpoint}", params={**defaults, **params}, timeout=15)
    r.raise_for_status()
    return r.json()

# Drucksachen
drucksachen = call_api("drucksache", **{"datum.start": "2024-01-01"})
print(f"Drucksachen: {len(drucksachen.get('documents', []))} gefunden")
for d in drucksachen["documents"]:
    print(f"  {d.get('datum', '?'):12}  {d.get('drucksachentyp', '?'):20}  {d.get('titel', '')[:60]}")

In [ ]:
# Personen / MdBs
personen = call_api("person", **{"fraktionMitgliedschaft.fraktion": "SPD"})
print(f"SPD-MdBs: {len(personen.get('documents', []))} gefunden")
for p in personen["documents"]:
    print(f"  {p.get('nachname', '?')}, {p.get('vorname', '?')}")